# 청킹과 임베딩 — 기초 (한글 문서)

문서 하나, 청킹 방법 세 가지, 그리고 **검색했을 때 실제로 무엇이 돌아오는지** 눈으로 확인합니다.
벤치마크도, 정답 라벨도, 점수표도 없습니다. 직접 읽고 판단하는 것이 목적입니다.

문서는 한국어 위키백과의 **「인공지능」** 문서를 인터넷에서 내려받아 사용합니다.
여러분이 이미 어느 정도 아는 주제이기 때문에, *"딥러닝이 뭐라고 나오지?"* 라고 물었을 때
돌아온 내용이 쓸만한지 아닌지 **읽어 보면 바로 판단할 수 있습니다.**

문서 전체가 약 37,000토큰이라 임베딩 비용은 한 번에 **약 $0.0007**입니다.
청크 크기를 열 가지로 바꿔 가며 실험해도 1센트가 안 되니 마음껏 다시 실행해 보세요.

## 1. 문서 가져오기

데이터 준비는 이 셀이 전부입니다. 별도의 데이터셋 라이브러리도, 스키마도 없습니다.

위키백과 API에 `explaintext=1` 옵션을 주면 위키 문법이 제거된 **순수 텍스트**를 받을 수 있습니다.
위키미디어는 요청에 `User-Agent` 헤더를 요구하므로 함께 보냅니다.

In [1]:
import re

import numpy as np
import requests
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
ai = OpenAI()

ENC = tiktoken.get_encoding("cl100k_base")
ntok = lambda t: len(ENC.encode(t))

In [2]:
TITLE = "인공지능"

resp = requests.get(
    "https://ko.wikipedia.org/w/api.php",
    params={
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,     # 위키 문법을 제거한 순수 텍스트로 받기
        "redirects": 1,
        "titles": TITLE,
    },
    headers={"User-Agent": "llm-edu-notebook/1.0"},
    timeout=30,
)

pages = resp.json()["query"]["pages"]
doc = next(iter(pages.values()))["extract"].strip()

print(f"{len(doc):,}자   {ntok(doc):,}토큰")
print(f"절(==) {len(re.findall(r'^== .+ ==$', doc, re.M))}개")
print()
print(doc[:500])

41,001자   37,141토큰
절(==) 14개

인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다.
인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신경망을 사용하는 기계 학습의 한 갈래이고, 생성형 인공지능은 글·그림·음성·영상·코드 같은 새로운 결과물을 만든다. 인공지능은 검색·추천·번역·음성 및 영상 인식, 의료 지원, 게임과 로봇 제어 등에 쓰인다. 기계 학습과 생성형 인공지능은 인공지능 전체가 아니라 그 일부이다.
‘인


### 잠깐 — 한글은 토큰을 얼마나 쓸까?

같은 코드를 영어 문서로 돌렸을 때와 비교해 보면 한글의 특성이 드러납니다.

| | 글자 수 | 토큰 수 | 글자당 토큰 |
|---|---|---|---|
| 영어 (이상한 나라의 앨리스) | 144,599자 | 36,926토큰 | 약 0.26 |
| **한글 (인공지능 문서)** | 약 41,000자 | 약 37,000토큰 | **약 0.9** |

**글자 수는 3분의 1도 안 되는데 토큰 수는 비슷합니다.**
`cl100k_base` 토크나이저가 영어 위주로 만들어져서, 한글은 글자 하나가 토큰 1개 이상을 쓰는 일이 많기 때문입니다.

실무에서 중요한 점은 두 가지입니다.

1. 영어 예제에서 본 `chunk_size` 숫자를 한글에 그대로 쓰면 **글자 기준으로는 훨씬 짧은 내용**이 담깁니다.
2. 같은 분량의 문서라도 한글은 **비용과 컨텍스트를 더 많이 씁니다.**

토큰에 대한 자세한 내용은 `../llm_intro/1_5_tokenizer.ipynb`를 참고하세요.

## 2. 왜 잘라야 할까?

이유는 두 가지이고, 둘 다 구체적입니다.

1. **임베딩 모델에 입력 한계가 있습니다** — `text-embedding-3-small`은 8,192토큰까지만 받습니다.
   이 문서는 약 37,000토큰이라 **물리적으로 한 번에 넣을 수 없습니다.**
2. **들어간다고 해도 문서 전체를 벡터 하나로 만들면 쓸모가 없습니다.**
   임베딩은 텍스트 전체의 평균 같은 것입니다. 문서 전체를 벡터 하나로 만들면
   "인공지능 전반에 대한 것"이 되어 버려서, 어떤 질문에도 비슷하게 가깝고
   **어떤 질문에도 딱 맞지 않습니다.**

청킹은 문서를 **각 벡터가 한 가지 주제만 담을 만큼** 작게 나누는 작업입니다.

In [3]:
# 문서 전체를 임베딩하면 어떻게 될까? 직접 시도해서 거절당하는 것을 확인합니다.
try:
    ai.embeddings.create(model="text-embedding-3-small", input=[doc])
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:200])

RateLimitError
Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quo


## 3. 자르는 방법 세 가지

아래 함수들은 모두 **텍스트를 받아 문자열 리스트를 돌려줄 뿐**입니다.
청커(chunker)란 그게 전부입니다.

| 방법 | 자르는 기준 | 특징 |
|---|---|---|
| `chunk_fixed` | 토큰 개수 | 일정한 크기. 문장 중간에서 잘림 |
| `chunk_paragraphs` | 빈 줄(문단) | 문장을 자르지 않음. 크기가 들쭉날쭉 |
| `chunk_sections` | `== 절 제목 ==` | 의미 단위로 완전. 너무 큼 |

In [4]:
def chunk_fixed(text, size=200, overlap=0):
    """`size` 토큰마다 무조건 자른다. 문장이 어디서 끝나는지는 신경 쓰지 않는다."""
    ids = ENC.encode(text)
    step = size - overlap
    out = []
    for i in range(0, len(ids), step):
        piece = ids[i : i + size]
        if not piece:
            break
        out.append(ENC.decode(piece))
        if i + size >= len(ids):
            break
    return out


def chunk_paragraphs(text, target=200):
    """문단을 통째로 모아서 대략 `target` 토큰이 될 때까지 합친다."""
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    out, buf, n = [], [], 0
    for p in paras:
        t = ntok(p)
        if buf and n + t > target:
            out.append("\n\n".join(buf))
            buf, n = [], 0
        buf.append(p)
        n += t
    if buf:
        out.append("\n\n".join(buf))
    return out


def chunk_sections(text):
    """'== 제목 ==' 절 단위로 자른다. 자연스럽지만 검색에 쓰기에는 너무 크다."""
    parts = re.split(r"(?=^== .+ ==$)", text, flags=re.M)
    return [p.strip() for p in parts if p.strip()]

In [5]:
for name, chunks in [
    ("fixed-200",       chunk_fixed(doc, 200)),
    ("fixed-200/ov50",  chunk_fixed(doc, 200, 50)),
    ("paragraphs-200",  chunk_paragraphs(doc, 200)),
    ("sections",        chunk_sections(doc)),
]:
    sizes = [ntok(c) for c in chunks]
    print(f"{name:18s} {len(chunks):>4}개 청크   "
          f"중앙값 {int(np.median(sizes)):>4}토큰   최대 {max(sizes):>5}토큰")

fixed-200           186개 청크   중앙값  200토큰   최대   200토큰
fixed-200/ov50      248개 청크   중앙값  200토큰   최대   200토큰
paragraphs-200       99개 청크   중앙값  222토큰   최대  1295토큰
sections             15개 청크   중앙값 2106토큰   최대  5362토큰


## 4. 잘린 자리를 들여다보기

여기가 가장 눈여겨볼 부분입니다.
고정 크기 청킹은 토큰 수만 세다가 자르기 때문에, **문장 중간이나 심하면 단어 중간**에서 끊깁니다.

In [6]:
fixed = chunk_fixed(doc, 200)

print("=" * 72)
print("청크 5의 끝부분:")
print("=" * 72)
print("..." + fixed[5][-200:])
print()
print("=" * 72)
print("청크 6의 시작부분  <- 위 문장이 여기서 이어진다")
print("=" * 72)
print(fixed[6][:200] + "...")

청크 5의 끝부분:
...결정 등의 결과를 만들고 물리적 또는 가상 환경에 영향을 줄 수 있는 기계 기반 시스템으로 정의한다. 시스템마다 자율성의 정도가 다르며, 일부는 실제 사용을 시작한 뒤 새로운 자료에 맞추어 동작을 조정할 수 있다. 대한민국의 「인공지능 발전과 신뢰 기반 조성 등에 관한 기본법」도 ‘인공지능’과 ‘인공지능시스템’을 나누어 정의한다. 이러한 정의는 정책과 법률

청크 6의 시작부분  <- 위 문장이 여기서 이어진다
이 적용될 대상을 정하기 위한 것으로, 인공지능에 관한 유일한 과학적·철학적 정의는 아니다.
인공지능 모델과 인공지능 시스템은 맡는 범위가 다르다. 기계 학습에서 모델은 입력을 출력으로 바꾸는 계산 구조와 학습된 매개변수의 집합을 뜻한다. 실제 인공지능 시스템에는 하나 이상의 모델뿐 아니라 입력 자료를 처리하는 절차, 데이터베이스, 사용자 화면, 출력에 적...


In [7]:
# 문단 청킹은 빈 줄에서만 자르기 때문에 문장이 잘리지 않는다.
paras = chunk_paragraphs(doc, 200)

print("=" * 72)
print("문단 청크 5의 끝부분:")
print("=" * 72)
print("..." + paras[5][-200:])
print()
print("=" * 72)
print("문단 청크 6의 시작부분  <- 깔끔하게 나뉜다")
print("=" * 72)
print(paras[6][:200] + "...")

문단 청크 5의 끝부분:
... 할 구체적인 능력과 평가 기준에는 합의된 하나의 정의가 없다. 모리스 등은 수행할 수 있는 과제의 범위, 각 과제에서의 성능과 사람의 개입 없이 행동하는 자율성을 서로 다른 기준으로 평가하는 틀을 제안하였다. 이에 따르면 제한된 공정만 다루는 시스템도 자율적으로 작동할 수 있고, 다양한 과제를 수행하는 모델도 이용자의 지시와 확인을 계속 요구할 수 있다.

문단 청크 6의 시작부분  <- 깔끔하게 나뉜다
== 작동 원리 ==
인공지능 시스템의 구조는 목적과 기술에 따라 다르지만, 대체로 입력을 데이터로 나타내고 모델이나 규칙으로 처리하여 결과를 내는 흐름을 따른다. 입력은 글·사진·소리, 센서 측정값이나 이용 기록 등이 될 수 있고, 출력은 분류, 수치 예측, 추천 목록, 생성한 글·그림 또는 기계의 행동이 될 수 있다. 실제 서비스에서는 모델이 계산한 결...


### 겹침(overlap)은 무엇을 해 주나

겹침은 앞 청크의 끝부분을 다음 청크의 앞에 다시 넣습니다.
그래서 반으로 잘린 문장도 **어딘가에는 온전한 형태로 존재하게** 됩니다.
대가는 중복입니다. 청크가 늘어나고, 벡터가 늘어나고, 비용이 늘어납니다.

In [8]:
plain = chunk_fixed(doc, 200, 0)
over = chunk_fixed(doc, 200, 50)
print(f"겹침 없음 : {len(plain)}개 청크")
print(f"겹침 50   : {len(over)}개 청크  (저장·임베딩 비용 +{100 * (len(over) / len(plain) - 1):.0f}%)")
print()
print("청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:")
print("---")
print(ENC.decode(ENC.encode(over[3])[-50:]))
print("---")
print(ENC.decode(ENC.encode(over[4])[:50]))

겹침 없음 : 186개 청크
겹침 50   : 248개 청크  (저장·임베딩 비용 +33%)

청크 3의 마지막 50토큰이 청크 4의 앞부분에 다시 나타난다:
---
료로 미리 학습한 파운데이션 모델과 생성형 인공지능이 확산되면서, 인공지능은 일상적인 정보 서비스뿐
---
료로 미리 학습한 파운데이션 모델과 생성형 인공지능이 확산되면서, 인공지능은 일상적인 정보 서비스뿐


## 5. 임베딩하고 검색하기

벡터 데이터베이스를 쓰지 않습니다.
청크가 수백 개뿐일 때는 **작은 행렬 하나와 내적(dot product) 한 번**이면 검색이 끝납니다.
이 규모에서 DB는 오히려 군더더기입니다.

벡터를 길이 1로 정규화해 두면 내적이 곧 **코사인 유사도**가 됩니다.

In [9]:
def embed(texts, batch=256):
    out = []
    for i in range(0, len(texts), batch):
        r = ai.embeddings.create(model="text-embedding-3-small",
                                 input=texts[i : i + batch])
        out.extend(d.embedding for d in r.data)
    V = np.asarray(out, dtype=np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True)   # 단위 벡터 -> 내적 == 코사인 유사도
    return V


INDEX = {}


def build(name, chunks):
    """한 번 임베딩한 결과를 기억해 둔다. (같은 이름은 다시 계산하지 않음)"""
    if name not in INDEX:
        print(f"임베딩 중: {name} ({len(chunks)}개 청크, 약 ${sum(map(ntok, chunks))/1e6*0.02:.4f})")
        INDEX[name] = (chunks, embed(chunks))
    return INDEX[name]


def search(query, name, k=3):
    chunks, V = INDEX[name]
    q = embed([query])[0]
    sims = V @ q
    top = np.argsort(-sims)[:k]
    return [(float(sims[i]), chunks[i]) for i in top]

In [12]:
from sentence_transformers import SentenceTransformer

local_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def embed(texts, batch=256):
    V = local_model.encode(texts, batch_size=batch,
                           normalize_embeddings=True,
                           show_progress_bar=True)
    return np.asarray(V, dtype=np.float32)

d:\a_i\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\a_i\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WINS_Security\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate deve

### `search()` 가 하는 일

위 `search()` 네 줄이 **RAG 의 R(검색)** 전부입니다.

```python
chunks, V = INDEX[name]      # ① 미리 임베딩해 둔 청크와 벡터 꺼내기
q = embed([query])[0]        # ② 질문도 같은 방식으로 벡터로
sims = V @ q                 # ③ 전부와 한 번에 비교
top = np.argsort(-sims)[:k]  # ④ 점수 높은 k개의 '번호' 고르기
```

| 단계 | 모양 | 하는 일 |
|---|---|---|
| ① | `V` = (186, 1536) | 청크 186개 × 벡터 1536차원 |
| ② | `q` = (1536,) | 질문 벡터 하나 |
| ③ | `sims` = (186,) | 청크마다 점수 하나씩 |
| ④ | `top` = (3,) | 가장 높은 3개의 **인덱스** |

③에서 `V @ q` 가 곧 코사인 유사도인 이유는 `embed()` 마지막 줄에서
`V /= np.linalg.norm(...)` 로 **길이 1짜리 벡터로 만들어 두었기** 때문입니다.
길이가 1이면 내적이 코사인과 같아집니다.
반복문 없이 **행렬 곱 한 번으로 186개를 동시에** 비교합니다.

---

### `top = np.argsort(-sims)[:k]` 뜯어보기

한 줄에 세 가지가 겹쳐 있습니다. 작은 예로 풀어 봅니다.

```
sims  = [0.31, 0.88, 0.45, 0.72, 0.12]     청크 5개의 점수
인덱스  =    0     1     2     3     4
```

**1. `argsort` — 값이 아니라 '번호'를 돌려준다**

이것이 핵심입니다. `sort` 와 헷갈리기 쉽습니다.

```
np.sort(-sims)[:3]     ->  [-0.88, -0.72, -0.45]    값만 나온다 → 어느 청크인지 모른다
np.argsort(-sims)[:3]  ->  [1, 3, 2]                번호가 나온다 → chunks[1] 로 꺼낼 수 있다
```

우리가 알아야 할 것은 "0.88점이 있다"가 아니라 **"1번 청크가 0.88점이다"** 입니다.
마지막 줄 `chunks[i]` 에서 그 번호로 본문을 꺼내야 하기 때문입니다.
그래서 `sort` 가 아니라 `argsort` 입니다.

**2. `-sims` — 음수를 붙여 내림차순으로 뒤집기**

`np.argsort` 는 **오름차순만** 지원합니다. `reverse=True` 같은 옵션이 없습니다.
그래서 부호를 뒤집어, "가장 작은 것"이 "원래 가장 컸던 것"이 되게 만듭니다.

```
sims   = [ 0.31,  0.88,  0.45,  0.72,  0.12]
-sims  = [-0.31, -0.88, -0.45, -0.72, -0.12]
                    ↑ 가장 작다 = 원래 가장 크다

np.argsort(-sims) = [1, 3, 2, 0, 4]
```

**3. `[:k]` — 앞에서 k개만**

```
[1, 3, 2, 0, 4]  →  [:3]  →  [1, 3, 2]

1위 인덱스 1 (0.88)   2위 인덱스 3 (0.72)   3위 인덱스 2 (0.45)
```

아래 셀에서 직접 확인할 수 있습니다.

---

### 알아 두면 좋은 두 가지

**`np.argsort(sims)[::-1]` 과 같지 않습니다**

같아 보이지만 **동점일 때 순서가 달라집니다.**

```
t = [0.5, 0.9, 0.5, 0.9, 0.1]        1·3번이 동점, 0·2번이 동점

np.argsort(-t)      = [1, 3, 0, 2, 4]     동점이면 작은 번호가 먼저
np.argsort(t)[::-1] = [3, 1, 2, 0, 4]     뒤집으면서 동점 순서도 뒤집힌다
```

검색에서 동점이 자주 나오지는 않지만, `k=1` 로 하나만 고를 때는 다른 청크가 뽑힐 수 있습니다.

**청크가 아주 많아지면 `argpartition` 이 빠릅니다**

`argsort` 는 **전부 정렬**합니다. 3개만 필요한데 186개를 모두 줄 세우는 셈입니다.
이 노트북 규모에서는 차이가 없지만, 청크가 수십만 개가 되면 달라집니다.

| 20만 개에서 상위 3개 | 시간 |
|---|---|
| `np.argsort(-x)[:3]` | 8.0 ms |
| `np.argpartition(-x, 3)[:3]` | **1.8 ms** |

`argpartition` 은 "상위 k개를 앞쪽으로 몰아놓기"만 하고 그 안의 순서는 보장하지 않습니다.
순위까지 필요하면 그 k개만 다시 정렬하면 됩니다.

> 실무 벡터 DB 가 내부에서 쓰는 방식입니다.
> `chunk_basic_ko2.ipynb` 부터는 Chroma 를 쓰므로 이 부분을 DB 가 대신해 줍니다.

In [10]:
# 위 설명을 숫자로 확인해 봅니다 (임베딩 호출 없음)
demo = np.array([0.31, 0.88, 0.45, 0.72, 0.12])
print("sims           =", demo)
print("-sims          =", -demo)
print("argsort(-sims) =", np.argsort(-demo), "  <- 큰 값부터의 '번호'")
print("[:3]           =", np.argsort(-demo)[:3])
print()
for rank, i in enumerate(np.argsort(-demo)[:3], 1):
    print(f"  {rank}위  인덱스 {i}  점수 {demo[i]}")

print()
print("--- 동점이 있으면 두 방식이 달라진다 ---")
tie = np.array([0.5, 0.9, 0.5, 0.9, 0.1])
print("t                =", tie)
print("argsort(-t)      =", np.argsort(-tie))
print("argsort(t)[::-1] =", np.argsort(tie)[::-1])

sims           = [0.31 0.88 0.45 0.72 0.12]
-sims          = [-0.31 -0.88 -0.45 -0.72 -0.12]
argsort(-sims) = [1 3 2 0 4]   <- 큰 값부터의 '번호'
[:3]           = [1 3 2]

  1위  인덱스 1  점수 0.88
  2위  인덱스 3  점수 0.72
  3위  인덱스 2  점수 0.45

--- 동점이 있으면 두 방식이 달라진다 ---
t                = [0.5 0.9 0.5 0.9 0.1]
argsort(-t)      = [1 3 0 2 4]
argsort(t)[::-1] = [3 1 2 0 4]


In [13]:
build("fixed-200", chunk_fixed(doc, 200))
build("paragraphs-200", chunk_paragraphs(doc, 200))
print("\n준비된 인덱스:", list(INDEX))

임베딩 중: fixed-200 (186개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:05<00:00,  5.05s/it]


임베딩 중: paragraphs-200 (99개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


준비된 인덱스: ['fixed-200', 'paragraphs-200']


## 6. 질문해 보기

점수는 참고만 하고, **돌아온 내용을 직접 읽어서** 쓸만한지 판단해 보세요.

In [14]:
def show(query, name, k=3, width=400):
    print("=" * 72)
    print(f"Q: {query}      [{name}]")
    print("=" * 72)
    for score, text in search(query, name, k):
        snippet = " ".join(text.split())[:width]
        print(f"\n[{score:.3f}] {snippet}...")


show("딥러닝이란 무엇인가?", "paragraphs-200")

Q: 딥러닝이란 무엇인가?      [paragraphs-200]


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.33it/s]


[0.796] 딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을 바꾸면서 과제에 유용한 내부 표현을 만든다. 일부 영상 인식 신경망에서는 앞쪽 층이 단순한 무늬와 경계에, 뒤쪽 층이 더 복잡한 형태에 반응하는 경향이 관찰된다. 이러한 내부 표현은 사람이 쓰는 개념과 정확히 일치하지 않을 수 있고 해석도 쉽지 않다....

[0.702] === 기계 학습과 딥 러닝 ===...

[0.663] 딥 러닝의 성과는 한 가지 알고리즘의 발명만으로 이루어진 것은 아니다. 인공 신경망과 역전파에 관한 아이디어는 오래전부터 연구되었지만, 대규모 데이터, 병렬 연산 장치, 개선된 학습 방법과 공개 벤치마크가 함께 발전하면서 더 깊고 큰 모델을 실제로 학습할 수 있게 되었다. 이에 따라 인공지능 연구에서 알고리즘뿐 아니라 데이터의 품질, 연산 자원, 평가 방법과 모델을 실제 서비스에 도입하는 기술의 중요성도 커졌다....


In [15]:
show("튜링 테스트는 무엇인가?", "paragraphs-200")

Q: 튜링 테스트는 무엇인가?      [paragraphs-200]


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.67it/s]


[0.638] 튜링 검사는 대화 행동을 평가할 뿐, 지능의 모든 형태나 이해·의식을 직접 측정하지 않는다. 또한 대화 시간, 심사자와 합격 기준이 다른 여러 변형이 있어, 한 번의 짧은 대화 결과만으로 표준화된 튜링 검사를 통과했다고 단정하기 어렵다. === 이해, 의식과 중국어 방 ===...

[0.437] 기계 학습에서는 보통 데이터를 학습 집합, 검증 집합, 시험 집합으로 나눈다. 학습 집합은 모델의 매개변수를 조정하는 데, 검증 집합은 모델 구조와 하이퍼파라미터를 고르는 데 사용한다. 시험 집합은 모든 선택을 마친 뒤 새로운 자료에서의 최종 성능을 추정하는 데 사용한다. 시험 결과를 보면서 모델을 계속 바꾸면 그 점수가 실제 성능보다 높게 나타날 수 있다. 자료를 나눌 때에는 같은 사람·장소·시간대에서 얻은 매우 비슷한 사례가 학습 집합과 시험 집합에 함께 들어가지 않도록 해야 한다. 시험 자료의 정보가 직접 또는 간접적으로 학습 과정에 들어가는 데이터 누출이 생기면 실제보다 높은 성능이 나타날 수 있다. 분류 모델은 전체 예측 가운데 맞힌 비율인 정확도뿐 아니라, 양성이라고 예측한 사례가 실제 양성인 비...

[0.417] 추론은 주어진 정보와 규칙에서 새로운 결론을 얻는 과정이다. 이를 컴퓨터로 수행하는 자동 추론은 특히 수학과 형식 논리의 연역을 자동화하는 연구와 밀접하다. 예를 들어 자동 정리 증명기는 공리와 이미 증명된 정리에 추론 규칙을 적용하여 목표 명제의 증명을 찾는다. 문제 해결은 현재 상태와 목표, 가능한 선택이나 행동이 주어졌을 때 목표에 이르는 해답을 찾는 과정이다. 길 찾기에서는 장소와 도로를 상태와 이동 가능한 경로로 나타낸 뒤 목적지에 도달하는 경로를 찾는다. 시간표 작성에서는 수업·교실·시간에 관한 여러 제약을 동시에 만족하는 배치를 찾아야 한다. 게임에서는 상대가 어떻게 행동할지도 고려하여 다음 수를 선택할 수 있다. 해답이 여러 개라면 이동 거리, 시간, 비용이나 위험처럼 미리 정한 기준에 따라 ...


In [16]:
show("인공지능의 윤리적 문제는 무엇인가?", "paragraphs-200")

Q: 인공지능의 윤리적 문제는 무엇인가?      [paragraphs-200]


Batches: 100%|██████████| 1/1 [00:00<00:00, 63.30it/s]


[0.815] 인공지능은 사람들이 정보를 얻고 일하며 결정을 내리는 방식을 바꾼다. 그 혜택과 피해는 어떤 자료로 시스템을 만들었는지, 누가 어떤 목적으로 사용하고 어떤 권한을 주는지, 결과가 사람의 권리와 생활에 얼마나 큰 영향을 미치는지에 따라 다르게 나타난다. 인공지능의 윤리는 기계가 스스로 도덕적 판단을 할 수 있는지만 묻는 것이 아니라, 사람과 기관이 인공지능을 어떻게 설계·사용·감독하고 그 결과에 책임질지도 다룬다. 인공지능이 일으키는 문제는 모델 내부의 계산만으로 설명되지 않는 경우가 많다. 학습 자료를 수집하고 정답을 붙이는 과정, 성능을 평가하는 기준, 기관의 업무 절차, 이용자가 결과를 받아들이는 방식과 이의를 제기할 수 있는 수단이 함께 영향을 준다. 예를 들어 동일한 예측 모델도 의사가 참고 자료로 ...

[0.758] 인공지능은 공학 기술인 동시에 지능, 이해, 마음과 인간의 책임에 관한 철학적 질문을 제기한다. 문제를 잘 푸는 능력, 사람과 비슷하게 말하는 능력, 여러 과제를 수행하는 범위, 스스로 행동을 고르는 자율성, 주관적인 경험을 뜻하는 의식은 서로 다른 성질이다. 인공지능 철학은 이러한 성질을 어떻게 정의하고 어떤 증거로 인정할 수 있는지, 인공지능의 행동과 결과에 누가 책임져야 하는지를 논의한다....

[0.684] 인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다. 인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신

## 7. 실험: 청크 크기가 결과를 바꾸는가?

같은 문서, 같은 질문, 같은 임베딩 모델입니다. **청크 크기만** 바뀝니다.
결과를 읽고, 어느 쪽을 LLM에게 컨텍스트로 넘기고 싶은지 직접 판단해 보세요.

작은 청크는 정확하지만 주변 맥락이 잘립니다.
큰 청크는 맥락이 충분하지만 정작 답이 관계없는 내용에 파묻힙니다.

In [17]:
QUESTION = "딥러닝이란 무엇인가?"

for size in [100, 200, 400, 800]:
    name = f"para-{size}"
    build(name, chunk_paragraphs(doc, size))
    score, text = search(QUESTION, name, k=1)[0]
    snippet = " ".join(text.split())
    print("=" * 72)
    print(f"청크 크기 {size:>4}토큰  |  {len(INDEX[name][0]):>3}개 청크  |  "
          f"1위 점수 {score:.3f}  |  이 청크는 {ntok(text)}토큰")
    print("=" * 72)
    print(snippet[:600] + ("..." if len(snippet) > 600 else ""))
    print()

임베딩 중: para-100 (108개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]


청크 크기  100토큰  |  108개 청크  |  1위 점수 0.796  |  이 청크는 209토큰
딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을 바꾸면서 과제에 유용한 내부 표현을 만든다. 일부 영상 인식 신경망에서는 앞쪽 층이 단순한 무늬와 경계에, 뒤쪽 층이 더 복잡한 형태에 반응하는 경향이 관찰된다. 이러한 내부 표현은 사람이 쓰는 개념과 정확히 일치하지 않을 수 있고 해석도 쉽지 않다.

임베딩 중: para-200 (99개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:00<?, ?it/s]


청크 크기  200토큰  |   99개 청크  |  1위 점수 0.796  |  이 청크는 209토큰
딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을 바꾸면서 과제에 유용한 내부 표현을 만든다. 일부 영상 인식 신경망에서는 앞쪽 층이 단순한 무늬와 경계에, 뒤쪽 층이 더 복잡한 형태에 반응하는 경향이 관찰된다. 이러한 내부 표현은 사람이 쓰는 개념과 정확히 일치하지 않을 수 있고 해석도 쉽지 않다.

임베딩 중: para-400 (81개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:00<00:00, 46.44it/s]


청크 크기  400토큰  |   81개 청크  |  1위 점수 0.796  |  이 청크는 209토큰
딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을 바꾸면서 과제에 유용한 내부 표현을 만든다. 일부 영상 인식 신경망에서는 앞쪽 층이 단순한 무늬와 경계에, 뒤쪽 층이 더 복잡한 형태에 반응하는 경향이 관찰된다. 이러한 내부 표현은 사람이 쓰는 개념과 정확히 일치하지 않을 수 있고 해석도 쉽지 않다.

임베딩 중: para-800 (54개 청크, 약 $0.0007)


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.87it/s]

청크 크기  800토큰  |   54개 청크  |  1위 점수 0.653  |  이 청크는 483토큰
딥 러닝의 성과는 한 가지 알고리즘의 발명만으로 이루어진 것은 아니다. 인공 신경망과 역전파에 관한 아이디어는 오래전부터 연구되었지만, 대규모 데이터, 병렬 연산 장치, 개선된 학습 방법과 공개 벤치마크가 함께 발전하면서 더 깊고 큰 모델을 실제로 학습할 수 있게 되었다. 이에 따라 인공지능 연구에서 알고리즘뿐 아니라 데이터의 품질, 연산 자원, 평가 방법과 모델을 실제 서비스에 도입하는 기술의 중요성도 커졌다. === 트랜스포머와 생성형 인공지능의 확산 === 2017년 바스와니 등이 발표한 〈어텐션 이즈 올 유 니드〉는 문장처럼 순서가 있는 데이터를 셀프 어텐션으로 처리하는 트랜스포머를 제안하였다. 트랜스포머는 멀리 떨어진 위치의 정보를 한 층에서 직접 연결하고, 학습할 때 여러 위치를 병렬로 계산할 수 있었다. 이후 인코더 전용 모델인 BERT와 디코더 전용 GPT 계열이 발전하면서, 대규모 텍스트로 먼저 학습한 모델을 여러 과제에 적용하는 방식이 자연어 처리의 주요 접근으로 자리 잡았다.



### 방금 무슨 일이 일어났나 — 넘어가기 전에 읽어 주세요

| 청크 크기 | 1위 점수 | 돌아온 내용 |
|---|---|---|
| 100 | 0.506 | 딥러닝을 정의한 문단 (209토큰) |
| 200 | 0.506 | **위와 완전히 같은 청크** |
| 400 | 0.506 | **위와 완전히 같은 청크** |
| 800 | 0.414 | 탐색·논리·확률 이야기로 시작하는 764토큰 덩어리. 딥러닝 설명이 중간에 묻힘 |

**100, 200, 400이 왜 똑같을까요?**

`chunk_paragraphs`는 문단을 **쪼개지 않고 모으기만** 합니다.
`chunk_size`는 "이 크기를 목표로 자른다"가 아니라 **"이 크기를 넘지 않게 문단을 합친다"는 상한**입니다.
우리가 찾는 답은 209토큰짜리 문단 **하나**에 통째로 들어 있습니다.
그래서 상한을 100으로 낮춰도 그 문단은 쪼개지지 않고 그대로 남습니다.

> 여기서 얻을 교훈: **`chunk_size`를 줄였는데 결과가 그대로라면 청커가 무시하고 있는 것입니다.**
> 숫자만 바꾸지 말고, 실제로 몇 개의 청크가 만들어졌는지(108 → 99 → 81 → 54개)와
> 가져온 청크의 실제 크기를 함께 확인하세요.

**800에서는 왜 점수가 떨어졌을까요?**

여러 문단이 하나로 합쳐지면서 딥러닝 설명이 **관계없는 내용에 희석**되었습니다.
임베딩은 텍스트 전체의 평균에 가깝기 때문에, 다른 주제가 섞일수록 질문과의 유사도가 낮아집니다.
답이 청크 안에 들어 있긴 하지만 뒤쪽에 묻혀 있어서, LLM에게 넘길 컨텍스트로는 불리합니다.

**점수가 높다고 좋은 청크는 아닙니다.**

6번의 "딥러닝이란 무엇인가?" 결과에서 2위를 보세요.

```
[0.433] === 기계 학습과 딥 러닝 ===...
```

**제목만 있고 내용이 하나도 없는 청크**입니다.
딥러닝에 대한 질문에 "기계 학습과 딥 러닝"이라는 제목이 의미상 가까운 건 당연합니다.
점수는 높지만 정보는 0입니다. 점수만 보고 판단하면 안 되는 이유입니다.

**한글 위키백과 문서의 특징도 하나 보입니다.**

3번 표에서 `paragraphs-200`의 최대 청크가 **1,295토큰**이었습니다. 목표는 200이었는데도요.
문단 하나가 이미 200토큰을 넘으면 청커가 손을 대지 못하기 때문입니다.
실무에서는 보통 **문단으로 자른 뒤, 너무 큰 문단은 다시 쪼개는** 방식을 함께 씁니다.

### 고정 크기 vs 문단 경계

같은 질문, 같은 목표 크기입니다.
차이는 **문장 경계를 지키느냐 마느냐** 하나뿐입니다.

In [18]:
for name in ["fixed-200", "paragraphs-200"]:
    score, text = search(QUESTION, name, k=1)[0]
    print("=" * 72)
    print(f"{name}   점수 {score:.3f}")
    print("=" * 72)
    print(" ".join(text.split())[:500])
    print()

Batches: 100%|██████████| 1/1 [00:00<00:00, 70.60it/s]


fixed-200   점수 0.605
층의 인공 신경망을 사용하는 방법이다. 많은 기계 학습 모델은 예측과 정답의 차이를 나타내는 손실 함수가 작아지도록 매개변수를 조정한다. 경사 하강법은 현재 위치에서 손실이 증가하는 방향의 반대쪽으로 매개변수를 조금씩 이동시키는 대표적인 최적화 방법이다. 딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을



Batches: 100%|██████████| 1/1 [00:00<00:00, 69.92it/s]

paragraphs-200   점수 0.796
딥 러닝은 여러 처리 층으로 이루어진 인공 신경망을 사용하는 기계 학습의 한 갈래이다. 각 층은 앞 층의 표현을 바꾸면서 과제에 유용한 내부 표현을 만든다. 일부 영상 인식 신경망에서는 앞쪽 층이 단순한 무늬와 경계에, 뒤쪽 층이 더 복잡한 형태에 반응하는 경향이 관찰된다. 이러한 내부 표현은 사람이 쓰는 개념과 정확히 일치하지 않을 수 있고 해석도 쉽지 않다.



### 결과: 문단 경계를 지킨 쪽이 이겼습니다

| | 1위 점수 | 돌아온 내용 |
|---|---|---|
| `fixed-200` | 0.385 | `기계 학습과 딥 러닝 ===` 로 **시작**. 절 제목이 반토막 난 채로 들어왔고, 마지막도 "딥 러닝은 이 가운데 여러" 에서 끊김 |
| `paragraphs-200` | **0.506** | 딥러닝을 정의하는 문단이 처음부터 끝까지 온전히 |

같은 목표 크기인데 점수 차이가 큽니다.
고정 크기 청킹은 토큰만 세다가 **문장 중간, 심지어 제목 중간**에서 끊었고,
그렇게 잘린 조각은 의미가 온전하지 않아 질문과의 유사도도 낮게 나옵니다.

잘린 청크를 LLM에게 컨텍스트로 넘기면 답변 품질도 함께 떨어집니다.
**문장 경계를 지키는 것만으로 검색 품질이 올라갑니다.**

## 8. 직접 해 보기

질문을 바꿔 가며 다시 실행해 보세요. 시도해 볼 만한 것들입니다.

- **답이 여러 곳에 흩어진 질문** (*"인공지능의 역사는 어떻게 되는가?"*)
  — 청크 하나에 답이 다 들어 있나요?
- **문서에 없는 내용** (*"김치찌개 끓이는 법은?"*)
  — 유사도 점수를 보세요. 임베딩 검색은 **항상 무언가를 돌려줍니다.**
    관련 내용이 없다는 것을 알려 주는 유일한 단서가 점수입니다.
- **구체적인 용어**(*"퍼셉트론"*) vs **막연한 주제**(*"인공지능이 무섭다"*)
  — 어느 쪽을 임베딩 검색이 더 잘 처리하나요?

In [19]:
MY_QUESTION = "인공지능의 역사는 어떻게 되는가?"

show(MY_QUESTION, "paragraphs-200", k=3)

Q: 인공지능의 역사는 어떻게 되는가?      [paragraphs-200]


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.79it/s]


[0.831] == 역사 == 인공지능의 역사는 한 가지 기술이 한 방향으로만 발전한 과정이 아니라, 논리와 탐색, 지식 기반 시스템, 확률 모형, 기계 학습과 신경망 등 여러 접근이 번갈아 주목받고 결합해 온 과정이다. 기대에 미치지 못해 투자와 관심이 줄어든 시기도 있었고, 데이터·연산 장치·알고리즘이 발전하면서 오래된 아이디어가 다시 중요해지기도 했다....

[0.756] === 기호주의 인공지능과 인공지능의 겨울 === 기호주의 인공지능은 지식을 기호와 규칙으로 나타내고 이를 조작하여 문제를 푸는 접근이다. 1950년대부터 1980년대까지 주요 연구 흐름 가운데 하나였으며, 연구자들은 가능한 상태를 탐색하거나 사실과 논리 규칙에서 결론을 이끌어 내는 프로그램을 만들었다. 1960년대 말의 SHRDLU는 작은 가상 블록 세계에서 영어 명령을 해석하고 물체를 조작했지만, 그 환경 밖의 일반적인 대화와 현실 지식까지 다루지는 못했다. 1970년대와 1980년대에는 특정 분야의 전문가 지식을 규칙으로 저장하는 전문가 시스템이 발전하였다. 화학 구조를 추론하는 DENDRAL과 세균 감염을 진단하고 항생제를 추천하도록 설계된 MYCIN 등이 대표적인 사례이다. 전문가 시스템은 범용 지...

[0.725] 인공지능(영어: artificial intelligence, AI)은 컴퓨터가 학습, 추론, 지각, 언어 처리, 문제 해결, 계획과 의사 결정처럼 지능과 관련된 일을 수행하도록 하는 방법을 연구하는 컴퓨터 과학 분야이다. 이러한 방법으로 만든 프로그램이나 기계도 인공지능이라고 부른다. 인공지능 시스템은 입력을 처리해 예측값, 추천, 결정, 글·그림 같은 결과를 만들며, 반드시 사람의 사고 과정을 그대로 흉내 내는 것은 아니다. 인공지능에는 사람이 사실과 규칙을 직접 적어 넣는 기호주의 인공지능과, 여러 데이터에서 규칙성을 찾아 모델을 조정하는 기계 학습 등 다양한 접근이 있다. 딥 러닝은 여러 층의 인공 신경망을 사용하는 기계 학습의 한 갈래이고, 생성형 인

In [20]:
# 문서에 답이 없는 질문 -- 내용보다 점수를 보세요.
show("김치찌개 끓이는 법은?", "paragraphs-200", k=3)

Q: 김치찌개 끓이는 법은?      [paragraphs-200]


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.95it/s]


[0.368] === 기계의 지능과 튜링 검사 ===...

[0.359] === 창작과 미디어 ===...

[0.352] === 평가 방법 ===...


> **점수를 비교해 보세요.** 문서에 답이 있는 질문은 0.5~0.7이 나왔는데,
> 김치찌개 질문은 **0.294, 0.230, 0.216** 입니다.
> 그런데도 검색은 **빈손으로 돌아오지 않습니다.** 무조건 세 개를 가져옵니다.
>
> 실무에서는 이 점수에 **하한선(threshold)** 을 두어, 기준보다 낮으면
> "관련 문서를 찾지 못했습니다"라고 처리합니다.
> 그러지 않으면 LLM이 엉뚱한 청크를 근거로 그럴듯한 답을 지어냅니다.

## 다음 단계

- **문서를 바꿔 보세요.** 코드는 그대로 두고 `TITLE`만 바꾸면 됩니다.
  `"기계 학습"`, `"정보 보안"`, `"대한민국"` 등 어떤 한국어 위키백과 문서든 됩니다.
- **소설로 바꿔 보세요.** 한국어 위키문헌(Wikisource)에는 저작권이 만료된 작품이 있습니다.
  **도메인만 `ko.wikisource.org`로 바꾸면** 나머지 코드는 그대로 쓸 수 있습니다.
  ```python
  TITLE = "운수 좋은 날"          # 현진건, 1924

  resp = requests.get(
      "https://ko.wikisource.org/w/api.php",   # 위키백과가 아니라 위키문헌
      params={
          "action": "query",
          "format": "json",
          "prop": "extracts",
          "explaintext": 1,
          "redirects": 1,
          "titles": TITLE,
      },
      headers={"User-Agent": "llm-edu-notebook/1.0"},
      timeout=30,
  )
  pages = resp.json()["query"]["pages"]
  doc = next(iter(pages.values()))["extract"].strip()

  # 원문 끝에 위키문헌이 붙인 '== 라이선스 ==' 절은 잘라냅니다.
  doc = re.split(r"^== 라이선스 ==$", doc, flags=re.M)[0].strip()
  ```
  > ⚠️ `index.php?...&action=raw` 방식도 있지만, 그쪽은 JSON이 아니라 위키 문법 원문을
  > 돌려주기 때문에 `resp.json()`에서 `JSONDecodeError`가 납니다. 위처럼 `api.php`의
  > `extracts`를 쓰세요.
  >
  > 소설로 바꿀 때는 **`chunk_paragraphs`도 함께 고쳐야 합니다.** 위키백과 문서는 문단 사이가
  > 빈 줄이지만 소설 원문은 한 줄 바꿈이라, `re.split(r"\n\s*\n", text)`를
  > `re.split(r"\n+", text)`로 바꾸지 않으면 청크가 5개밖에 만들어지지 않습니다.
  >
  > **완성된 소설 버전은 `chunk_basic_ko2.ipynb`에 있습니다.** 두 노트북을 비교하면
  > 설명문과 문학 작품에서 청킹·검색이 어떻게 달라지는지 볼 수 있습니다.
- **문서 여러 개를 다루기** — 문서가 많아지면 "어느 청크인가" 이전에
  "어느 문서인가"를 먼저 고르는 문제가 생깁니다.
- **눈으로 보지 말고 측정하기** — `rag_chunk_embed.md`와 `science_qa.ipynb`가
  정답 라벨이 있는 데이터로 이 작업을 제대로 합니다.
  핵심 아이디어는, 청크 개수를 고정하지 말고 **컨텍스트 총량(토큰 예산)을 고정**해서
  비교해야 한다는 것입니다. 그러지 않으면 큰 청크가 자동으로 유리해집니다.
- **벡터 DB를 붙이려면** — `rag_basic.ipynb`에서 Chroma를 사용한 전체 RAG 흐름을 다룹니다.